In [37]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [3]:
dataset = pd.read_csv("./train.csv", index_col="Id")

In [56]:
dataset = dataset.dropna(subset=["SalePrice"]).reset_index(drop=True)

X = dataset.iloc[:, :-1]
y = dataset.iloc[:, -1]

Rating colums with values:
* Ex (Excellent)
* Gd (Good)
* TA (Average/Typical)
* Fa (Fair)
* Po (Poor)
* NA

In [57]:
QUAL_COND_COLUMS = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]
qual_cond_map = {"Ex" : 6, "Gd": 5, "TA" : 4, "Fa": 3, "Po" : 2, "NA" : 1}
X[QUAL_COND_COLUMS] = X[QUAL_COND_COLUMS].replace(qual_cond_map)

Utilities: Type of utilities available
* AllPub	All public Utilities (E,G,W,& S)	
* NoSewr	Electricity, Gas, and Water (Septic Tank)
* NoSeWa	Electricity and Gas Only
* ELO	Electricity only	

In [43]:
util_map = {"AllPub": {"Electricity": 1, "Gas": 1, "Water": 1, "Septic": 1},
            "NoSewr": {"Electricity": 1, "Gas": 1, "Water": 1, "Septic": 0},
            "NoSeWa": {"Electricity": 1, "Gas": 1, "Water": 0, "Septic": 0},
            "ELO": {"Electricity": 1, "Gas": 0, "Water": 0, "Septic": 0}
            }

util_colums = pd.DataFrame(list(X["Utilities"].map(util_map)))
X = X.join(util_colums).drop(columns=["Utilities"])

In [58]:
CATEGORICAL_VALUES = ["MSZoning", "Street", "Alley", "LotShape", "LandContour", "LotConfig", "LandSlope", "Neighborhood", "Condition1", "Condition2", \
                      "BldgType", "HouseStyle", "RoofStyle", "RoofMatl", "Exterior1st", "Exterior2nd", "MasVnrType", "Foundation",  \
                      "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "Heating", "CentralAir", "Electrical", "Functional", \
                      "GarageType", "GarageFinish", "PavedDrive", "Fence", "MiscFeature", "SaleType", "SaleCondition"]

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded = encoder.fit_transform(X[CATEGORICAL_VALUES])
encoded_df = pd.DataFrame(encoded.todense(), columns=encoder.get_feature_names_out(CATEGORICAL_VALUES))
X = pd.concat([X.drop(columns=CATEGORICAL_VALUES), encoded_df], axis=1)

In [60]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [61]:
imputer = SimpleImputer(strategy="most_frequent")
imputed = imputer.fit_transform(X_train)
X_train = pd.DataFrame(imputed, columns=imputer.get_feature_names_out())

In [62]:
NUMERICAL_VALUES = ["LotFrontage", "LotArea", "YearBuilt", "YearRemodAdd", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",\
                    "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea", "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath",\
                    "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt", "GarageCars", "GarageArea", "WoodDeckSF",\
                    "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "PoolArea", "MiscVal", "MoSold", "YrSold"]

In [63]:
scaler = StandardScaler()
X_train[NUMERICAL_VALUES] = scaler.fit_transform(X_train[NUMERICAL_VALUES])

In [64]:
reg = RandomForestRegressor(random_state=42)

In [65]:
reg.fit(X_train, y_train)

ValueError: could not convert string to float: 'AllPub'

In [51]:
new_imputed = imputer.transform(X_test)
X_test = pd.DataFrame(new_imputed, columns=imputer.get_feature_names_out())
X_test[NUMERICAL_VALUES] = scaler.transform(X_test[NUMERICAL_VALUES])

In [54]:
y_pred = reg.predict(X_test)

In [55]:
reg.score(X_test, y_test)

0.8904376677803645